In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer, normalize

import umap
import hdbscan

In [ ]:
data = np.load("user_embeddings.npz", allow_pickle=True)
embeddings = data["embeddings"]
user_names = data["user_names"]

In [10]:
df_emb = pd.DataFrame({
    "user_name": user_names,
    "embedding": list(embeddings)   # jede Zeile = 1 Array
})

In [ ]:
df_users = pd.read_parquet("bitcoin_users.parquet", columns=['user_name', 'user_followers', 'user_friends', 'user_favourites', 'user_verified', 'hashtags', 'total_tweets', 'retweet_ratio'])

In [12]:
df_tocluster = df_users.merge(df_emb, on="user_name", how="inner")

In [13]:
df_tocluster.columns

Index(['user_name', 'text', 'user_description', 'user_followers',
       'user_friends', 'user_favourites', 'user_verified', 'hashtags',
       'retweet_count', 'total_tweets', 'retweet_ratio', 'embedding'],
      dtype='object')

In [ ]:
X_emb = np.stack(df_tocluster['embedding'].values)

X_emb_norm = normalize(X_emb, norm='l2')

scaler_emb = StandardScaler()
X_emb_scaled = scaler_emb.fit_transform(X_emb_norm)

In [ ]:
reducer_text = umap.UMAP(
    n_neighbors=15, 
    min_dist=0.1, 
    n_components=50,       # z.B. 10-dim für HDBSCAN
    metric='cosine',       # für Text-Embeddings sinnvoll
    random_state=42
)

X_umap_emb = reducer_text.fit_transform(X_emb_scaled)

In [ ]:
mlb = MultiLabelBinarizer()
X_hashtags = mlb.fit_transform(df_tocluster['hashtags']) 

In [ ]:
reducer_tags = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=15,
    metric='cosine',
    random_state=42
)
X_umap_hashtag = reducer_tags.fit_transform(X_hashtags)

In [ ]:
X_meta = df_tocluster[['user_followers','user_friends','user_favourites','retweet_ratio','total_tweets','user_verified']].fillna(0)

scaler_meta = StandardScaler()
X_meta_scaled = scaler_meta.fit_transform(X_meta)

In [ ]:
X_combined = np.hstack([X_umap_emb, X_umap_hashtag, X_meta_scaled])

In [ ]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=15000,   # große Cluster → weniger Gesamtcluster
    min_samples=50,           # Core-Punkte
    metric='euclidean',       # auf UMAP Euclidean
    cluster_selection_method='eom',
    prediction_data=True      # nötig für cluster_probs
)

cluster_labels = clusterer.fit_predict(X_combined)
cluster_probs = clusterer.probabilities_

In [ ]:
is_exemplar = np.zeros(len(cluster_labels), dtype=bool)

exemplars = clusterer.exemplars_

# dict → Werte nehmen, liste → direkt
if isinstance(exemplars, dict):
    iterable = exemplars.values()
else:
    iterable = exemplars

for group in iterable:
    # group kann sein: int, array([idx]), array([[idx]]), array([idx1, idx2]), ...
    group = np.array(group).flatten()   # sauber machen

    for idx in group:
        idx = int(idx)  # float → int

        if 0 <= idx < len(is_exemplar):
            is_exemplar[idx] = True

In [ ]:
df_user_cluster_assignment = pd.DataFrame({
    'user_name': df_tocluster['user_name'].values,
    'cluster': cluster_labels,
    'cluster_prob': cluster_probs,
    'is_exemplar': is_exemplar
})

In [ ]:
df_user_cluster_assignment["cluster"].value_counts()

In [ ]:
df_user_cluster_as
signment["is_exemplar"].value_counts()

In [ ]:
df_user_cluster_assignment.to_parquet("bitcoin_user_cluster_assignement.parquet", index=False, compression="zstd")